In [1]:
import os
import glob
import numpy as np
import pandas as pd

import librosa
import librosa.display

from tqdm import tqdm


import torch
from torch.utils.data import Dataset
from transformers import (
    AutoFeatureExtractor,
    AutoModelForAudioClassification,
    TrainingArguments,
    Trainer
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
# SECTION : Dataset Config

DATA_PATH = "/Users/cryptelle/Downloads/remake"

TARGET_SR = 16000
TARGET_SEC = 3
TARGET_LEN = TARGET_SR * TARGET_SEC

TOP_DB = 30
FRAME_LENGTH = 2048
HOP_LENGTH = 512

N_MELS = 128
N_FFT = 1024

In [3]:
# SECTION : Collect WAV Files

wav_paths = glob.glob(
    os.path.join(DATA_PATH, "**", "*.wav"),
    recursive=True
)

wav_files = [
    os.path.basename(f)
    for f in wav_paths
]

print("Total WAV files:", len(wav_paths))
print("First 10 paths:")
print(wav_paths[:10])

Total WAV files: 3870
First 10 paths:
['/Users/cryptelle/Downloads/remake/0/50-f-5-0-2-37.wav', '/Users/cryptelle/Downloads/remake/0/26-m-19-0-0-127.wav', '/Users/cryptelle/Downloads/remake/0/56-f-40-0-2-94.wav', '/Users/cryptelle/Downloads/remake/0/9-f-20-0-1-104.wav', '/Users/cryptelle/Downloads/remake/0/43-m-41-0-1-154.wav', '/Users/cryptelle/Downloads/remake/0/47-m-20-0-2-171.wav', '/Users/cryptelle/Downloads/remake/0/53-m-20-0-2-211.wav', '/Users/cryptelle/Downloads/remake/0/56-f-40-0-0-52.wav', '/Users/cryptelle/Downloads/remake/0/11-m-23-0-0-108.wav', '/Users/cryptelle/Downloads/remake/0/49-m-23-0-2-198.wav']


In [4]:
# SECTION : Labels

def get_emotion(fname):

    base = os.path.splitext(fname)[0]
    parts = base.split("-")

    return int(parts[4])   # spoken_emotion


df = pd.DataFrame({
    "file": wav_files,
    "path": wav_paths
})

df["y"] = df["file"].apply(get_emotion)

EMO_MAP = {
    0: "Low",
    1: "Neutral",
    2: "High"
}

df["emotion_name"] = df["y"].map(EMO_MAP)

df2 = df.copy()
df2["y"] = df2["y"].astype(int)

print(df2["emotion_name"].value_counts())
print(df2.head())

emotion_name
High       1356
Neutral    1344
Low        1170
Name: count, dtype: int64
                  file                                               path  y  \
0    50-f-5-0-2-37.wav  /Users/cryptelle/Downloads/remake/0/50-f-5-0-2...  2   
1  26-m-19-0-0-127.wav  /Users/cryptelle/Downloads/remake/0/26-m-19-0-...  0   
2   56-f-40-0-2-94.wav  /Users/cryptelle/Downloads/remake/0/56-f-40-0-...  2   
3   9-f-20-0-1-104.wav  /Users/cryptelle/Downloads/remake/0/9-f-20-0-1...  1   
4  43-m-41-0-1-154.wav  /Users/cryptelle/Downloads/remake/0/43-m-41-0-...  1   

  emotion_name  
0         High  
1          Low  
2         High  
3      Neutral  
4      Neutral  


In [5]:
# SECTION : Audio Preprocessing

def preprocess_audio(path):

    y, sr = librosa.load(
        path,
        sr=TARGET_SR,
        mono=True
    )

    # Remove silence
    y_trim, _ = librosa.effects.trim(
        y,
        top_db=TOP_DB,
        frame_length=FRAME_LENGTH,
        hop_length=HOP_LENGTH
    )

    # Safety fallback
    if len(y_trim) < 10:
        y_trim = y

    # Normalize
    peak = np.max(np.abs(y_trim)) + 1e-9

    y_trim = y_trim / peak

    # Fixed length
    if len(y_trim) > TARGET_LEN:

        y_trim = y_trim[:TARGET_LEN]

    else:

        y_trim = np.pad(
            y_trim,
            (0, TARGET_LEN - len(y_trim))
        )

    return y_trim

In [6]:
# SECTION : Build Waveforms

X_wave = np.zeros(
    (len(df2), TARGET_LEN),
    dtype=np.float32
)

for i, path in enumerate(
    tqdm(df2["path"], desc="Processing Audio")
):

    X_wave[i] = preprocess_audio(path)

print("X_wave shape:", X_wave.shape)

Processing Audio: 100%|████████████████████| 3870/3870 [00:05<00:00, 668.02it/s]

X_wave shape: (3870, 48000)


In [7]:
X_wave
df2

,file,path,y,emotion_name
0,50-f-5-0-2-37.wav,/Users/cryptelle/Downloads/remake/0/50-f-5-0-2...,2,High
1,26-m-19-0-0-127.wav,/Users/cryptelle/Downloads/remake/0/26-m-19-0-...,0,Low
2,56-f-40-0-2-94.wav,/Users/cryptelle/Downloads/remake/0/56-f-40-0-...,2,High
3,9-f-20-0-1-104.wav,/Users/cryptelle/Downloads/remake/0/9-f-20-0-1...,1,Neutral
4,43-m-41-0-1-154.wav,/Users/cryptelle/Downloads/remake/0/43-m-41-0-...,1,Neutral
...,...,...,...,...
3865,55-m-16-5-0-1560.wav,/Users/cryptelle/Downloads/remake/5/55-m-16-5-...,0,Low
3866,55-m-16-5-1-1574.wav,/Users/cryptelle/Downloads/remake/5/55-m-16-5-...,1,Neutral
3867,50-f-5-5-0-1377.wav,/Users/cryptelle/Downloads/remake/5/50-f-5-5-0...,0,Low
3868,56-f-40-5-2-1455.wav,/Users/cryptelle/Downloads/remake/5/56-f-40-5-...,2,High


In [8]:
# SECTION Augmentation


def add_noise(data, noise_factor=0.005):

    noise = np.random.randn(len(data))

    return data + noise_factor * noise


def shift_pitch(data, sr, n_steps=2):

    return librosa.effects.pitch_shift(
        data,
        sr=sr,
        n_steps=n_steps
    )


def time_shift(data, shift_max=0.2):

    shift = int(
        np.random.uniform(-shift_max, shift_max)
        * len(data)
    )

    return np.roll(data, shift)


def change_volume(data):

    gain = np.random.uniform(0.8, 1.2)

    return data * gain

In [9]:
# SECTION: Log-Mel Extraction


def waveform_to_logmel(y):

    S = librosa.feature.melspectrogram(
        y=y,
        sr=TARGET_SR,
        n_mels=N_MELS,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH
    )

    logS = librosa.power_to_db(
        S,
        ref=np.max
    )

    target_width = 94

    if logS.shape[1] < target_width:

        pad = target_width - logS.shape[1]

        logS = np.pad(
            logS,
            ((0,0),(0,pad)),
            mode='constant'
        )

    else:

        logS = logS[:, :target_width]

    return logS

In [10]:
# SECTION : Feature Extraction


X_mel_list = []
y_augmented = []

for i in tqdm(range(len(df2))):

    y_wave = X_wave[i]

    label = df2["y"].iloc[i]

    # Original
    mel = waveform_to_logmel(y_wave)

    X_mel_list.append(mel)
    y_augmented.append(label)

    # Noise
    noisy = add_noise(y_wave)

    mel_noise = waveform_to_logmel(noisy)

    X_mel_list.append(mel_noise)
    y_augmented.append(label)

    # Pitch Shift
    pitched = shift_pitch(
        y_wave,
        TARGET_SR
    )

    mel_pitch = waveform_to_logmel(pitched)

    X_mel_list.append(mel_pitch)
    y_augmented.append(label)

    # Time Shift
    shifted = time_shift(y_wave)

    mel_shift = waveform_to_logmel(shifted)

    X_mel_list.append(mel_shift)
    y_augmented.append(label)

    # Volume Change
    volume = change_volume(y_wave)

    mel_volume = waveform_to_logmel(volume)

    X_mel_list.append(mel_volume)
    y_augmented.append(label)


X = np.array(
    X_mel_list,
    dtype=np.float32
)

X = X[..., np.newaxis]

y = np.array(y_augmented)

print(X.shape)
print(y.shape)

100%|███████████████████████████████████████| 3870/3870 [01:55<00:00, 33.40it/s]


(19350, 128, 94, 1)
(19350,)


In [14]:
#

In [19]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Balanced K-Means using RMS Energy


rms_energy_list = []
zcr_list = []

for path in df2["path"]:
    y_audio, sr = librosa.load(path, sr=16000, mono=True)

    rms_energy = np.mean(librosa.feature.rms(y=y_audio))
    zcr = np.mean(librosa.feature.zero_crossing_rate(y_audio))

    rms_energy_list.append(rms_energy)
    zcr_list.append(zcr)

df2["rms_energy"] = rms_energy_list
df2["zcr"] = zcr_list

X_energy = df2[["rms_energy"]].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_energy)

kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=50
)

df2["cluster"] = kmeans.fit_predict(X_scaled)

cluster_stats = df2.groupby("cluster")[["rms_energy", "zcr"]].mean()
anxiety_cluster = cluster_stats["rms_energy"].idxmax()

df2["kmeans_label"] = df2["cluster"].apply(
    lambda c: "Anxiety-like" if c == anxiety_cluster else "Non-Anxiety"
)

print("\nCluster acoustic statistics:")
print(cluster_stats)

print("\nAssigned Anxiety-like cluster:", anxiety_cluster)

print("\nFinal K-Means labels:")
print(df2["kmeans_label"].value_counts())

print("\nK-Means label vs original emotion_name:")
print(pd.crosstab(df2["kmeans_label"], df2["emotion_name"]))


Cluster acoustic statistics:
         rms_energy       zcr
cluster                      
0          0.027532  0.124965
1          0.090717  0.142334

Assigned Anxiety-like cluster: 1

Final K-Means labels:
kmeans_label
Non-Anxiety     2920
Anxiety-like     950
Name: count, dtype: int64

K-Means label vs original emotion_name:
emotion_name  High   Low  Neutral
kmeans_label                     
Anxiety-like   640   120      190
Non-Anxiety    716  1050     1154
